# Faruq-v3 — LRLIN_FRESH seed 42
Capacity-matched low-rank linear residual control. Fresh end-to-end training from the SHA-locked official `yolo26n.pt`; no Coffee checkpoint, freeze, continuation, or test access. This notebook owns only `LRLIN_FRESH`.

In [ ]:
import torch
assert torch.cuda.is_available(), 'STOP CEPAT: aktifkan Runtime > Change runtime type > T4 GPU.'
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
ARM='LRLIN_FRESH'; BRANCH='codex/dlrbc-fresh-screening'
from google.colab import drive
drive.mount('/content/drive')
import csv, hashlib, importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone,cwd='/content')
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    time.sleep(2)
else: raise RuntimeError('Git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True,cwd='/content')
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT=resolve_drive_project_root(required_relative_paths=('bundles/faruq-development-v3-grouped.tar',))
ARCHIVE=require_project_artifact(PROJECT,'bundles/faruq-development-v3-grouped.tar')
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
GROUPED=DATA/'faruq_grouped_summary.json'
assert (DATA/'data.yaml').is_file() and GROUPED.is_file() and not (DATA/'test').exists(), 'Kontrak development/test gagal'
from ultralytics import YOLO
_=YOLO('yolo26n.pt'); PRETRAINED=(REPO/'yolo26n.pt').resolve(); assert PRETRAINED.is_file()
OUTPUT=PROJECT/'experiments/faruq-v3-dlrbc-fresh-v1'; OUTPUT.mkdir(parents=True,exist_ok=True)
STATIC=OUTPUT/'static'/f'{ARM}_seed42_static_audit.json'; STATIC.parent.mkdir(parents=True,exist_ok=True)
print('ARM:',ARM,'| PROJECT:',PROJECT,'| OUTPUT:',OUTPUT)

In [ ]:
from coffee_detector.experiments.run_faruq_v3_dlrbc_fresh_arm import run_fresh_static_audit
audit=run_fresh_static_audit(PRETRAINED,STATIC,seed=42,device='cpu')
print('STATIC:',audit['decision'],'| PARAMS:',audit['parameter_count'])
print('MATCHED ADAPTER SHA:',audit['adapter_initial_state_sha256'])
assert audit['decision']=='PASS' and audit['test_images_accessed'] is False, 'STOP: static audit gagal'

In [ ]:
LOG=OUTPUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_dlrbc_fresh_arm','--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--pretrained-checkpoint',str(PRETRAINED),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME SAME RUN:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
shown=None
while process.poll() is None:
    path=OUTPUT/ARM/f'{ARM}_seed42'/'results.csv'
    try:
        with path.open(newline='',encoding='utf-8') as handle: epochs=sum(1 for _ in csv.DictReader(handle))
    except Exception: epochs=0
    if epochs!=shown: print(f'{ARM}: {epochs}/50 epoch tercatat',flush=True); shown=epochs
    time.sleep(120)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-160:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
RESULT=OUTPUT/'val_reports'/f'{ARM}_seed42_result.json'; result=json.loads(RESULT.read_text(encoding='utf-8'))
print(json.dumps(result,indent=2,ensure_ascii=False)); print('Drive last.pt dapat dipakai resume untuk arm yang sama.')